# Data Ingestion And Source Validation

This notebook documents the first data step for the Medicaid Enrollment & Eligibility Operations Dashboard project. It uses official public CMS/Data.Medicaid.gov aggregate data only. No individual-level, claims-level, restricted, or unofficial scraped data are used.

Notebook questions:

1. What did I do?
2. Why did I do it?
3. What does it mean for healthcare reporting, Medicaid policy monitoring, or operational decision-making?
4. What are the data limitations?

## Source Selected

- Dataset source name: State Medicaid and CHIP Applications, Eligibility Determinations, and Enrollment Data
- Publisher: CMS/Data.Medicaid.gov
- Source URL: https://data.medicaid.gov/dataset/6165f45b-ca93-5bb5-9d06-db29c692a360/data
- Download URL used by script: https://download.medicaid.gov/data/pi-dataset-may-2026-release.csv
- Access method: public HTTPS CSV download plus CMS metadata/data dictionary endpoints
- Reporting grain: monthly state-level aggregate records

The source measures Medicaid/CHIP applications, eligibility determinations, enrollment, selected application processing timing fields, and selected call center fields. The cleaning step filters to 2019-present and all 50 states plus DC.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from clean_medicaid_data import (
    FULL_CLEAN_PATH,
    QUALITY_SUMMARY_PATH,
    SAMPLE_PATH,
    STATE_MONTH_SUMMARY_PATH,
    clean_data,
    save_outputs,
    standardize_column_names,
    parse_reporting_period,
    STATE_ABBREVIATIONS,
)
from load_data import RAW_CSV_PATH, download_source_csv, load_raw_data, save_source_metadata

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
print(PROJECT_ROOT)

/Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard


## Load Official Source Data

The script caches the official raw CSV and metadata under `data/raw/`. That folder is intentionally excluded from Git so the repository stays lightweight. The processed outputs are saved under `data/processed/`.

In [2]:
metadata_path = save_source_metadata()
csv_path = download_source_csv()
raw = load_raw_data(csv_path)

print(f"Metadata path: {metadata_path}")
print(f"Raw CSV path: {csv_path}")
print(f"Raw row count: {len(raw):,}")
print(f"Raw column count: {len(raw.columns):,}")
raw.head()

Metadata path: /Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard/data/raw/source_metadata.json
Raw CSV path: /Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard/data/raw/pi-dataset-may-2026-release.csv
Raw row count: 10,710
Raw column count: 44


,State Abbreviation,State Name,Reporting Period,State Expanded Medicaid,Preliminary or Updated,Final Report,New Applications Submitted to Medicaid and CHIP Agencies,New Applications Submitted to Medicaid and CHIP Agencies - footnotes,Applications for Financial Assistance Submitted to the State Based Marketplace,Applications for Financial Assistance Submitted to the State Based Marketplace - footnotes,Total Applications for Financial Assistance Submitted at State Level,Total Applications for Financial Assistance Submitted at State Level - footnotes,Individuals Determined Eligible for Medicaid at Application,Individuals Determined Eligible for Medicaid at Application - footnotes,Individuals Determined Eligible for CHIP at Application,Individuals Determined Eligible for CHIP at Application - footnotes,Total Medicaid and CHIP Determinations,Total Medicaid and CHIP Determinations - footnotes,Medicaid and CHIP Child Enrollment,Medicaid and CHIP Child Enrollment - footnotes,Total Medicaid and CHIP Enrollment,Total Medicaid and CHIP Enrollment - footnotes,Total Medicaid Enrollment,Total Medicaid Enrollment - footnotes,Total CHIP Enrollment,Total CHIP Enrollment - footnotes,Total Adult Medicaid Enrollment,Total Adult Medicaid Enrollment - footnotes,Total Medicaid and CHIP Determinations Processed in Less than 24 Hours,Total Medicaid and CHIP Determinations Processed in Less than 24 Hours - footnotes,Total Medicaid and CHIP Determinations Processed Between 24 Hours and 7 Days,Total Medicaid and CHIP Determinations Processed Between 24 Hours and 7 Days - footnotes,Total Medicaid and CHIP Determinations Processed Between 8 Days and 30 Days,Total Medicaid and CHIP Determinations Processed Between 8 Days and 30 Days - footnotes,Total Medicaid and CHIP Determinations Processed between 31 days and 45 days,Total Medicaid and CHIP Determinations Processed between 31 days and 45 days - footnotes,Total Medicaid and CHIP Determinations Processed in More than 45 Days,Total Medicaid and CHIP Determinations Processed in More than 45 Days - footnotes,Total Call Center Volume (Number of Calls),Total Call Center Volume (Number of Calls) - footnotes,Average Call Center Wait Time (Minutes),Average Call Center Wait Time (Minutes) - footnotes,Average Call Center Abandonment Rate,Average Call Center Abandonment Rate - footnotes
0,AK,Alaska,201309,N,U,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122334.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AK,Alaska,201706,Y,P,N,3017.0,Includes Renewals and/or Redeterminations,0.0,NaN,3017.0,Includes Renewals and/or Redeterminations,3422.0,Includes Renewals and/or Redeterminations; Inc...,0.0,NaN,3422.0,Includes Renewals and/or Redeterminations,88766.0,NaN,191171.0,NaN,179001.0,NaN,12170.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AK,Alaska,201706,Y,U,Y,3069.0,Includes Renewals and/or Redeterminations,0.0,NaN,3069.0,Includes Renewals and/or Redeterminations,3422.0,Includes Renewals and/or Redeterminations; Inc...,0.0,NaN,3422.0,Includes Renewals and/or Redeterminations,90081.0,NaN,194534.0,NaN,182080.0,NaN,12454.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AK,Alaska,201707,Y,P,N,3187.0,Includes Renewals and/or Redeterminations,0.0,NaN,3187.0,Includes Renewals and/or Redeterminations,3159.0,Includes Renewals and/or Redeterminations; Inc...,0.0,NaN,3159.0,Includes Renewals and/or Redeterminations,89358.0,NaN,193019.0,NaN,180641.0,NaN,12378.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AK,Alaska,201707,Y,U,Y,3265.0,Includes Renewals and/or Redeterminations,0.0,NaN,3265.0,Includes Renewals and/or Redeterminations,3159.0,Includes Renewals and/or Redeterminations; Inc...,0.0,NaN,3159.0,Includes Renewals and/or Redeterminations,90562.0,NaN,196121.0,NaN,183516.0,NaN,12605.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print("Source column names:")
for column in raw.columns:
    print(f"- {column}")

Source column names:
- State Abbreviation
- State Name
- Reporting Period
- State Expanded Medicaid
- Preliminary or Updated
- Final Report
- New Applications Submitted to Medicaid and CHIP Agencies
- New Applications Submitted to Medicaid and CHIP Agencies - footnotes
- Applications for Financial Assistance Submitted to the State Based Marketplace
- Applications for Financial Assistance Submitted to the State Based Marketplace - footnotes
- Total Applications for Financial Assistance Submitted at State Level
- Total Applications for Financial Assistance Submitted at State Level - footnotes
- Individuals Determined Eligible for Medicaid at Application
- Individuals Determined Eligible for Medicaid at Application - footnotes
- Individuals Determined Eligible for CHIP at Application
- Individuals Determined Eligible for CHIP at Application - footnotes
- Total Medicaid and CHIP Determinations
- Total Medicaid and CHIP Determinations - footnotes
- Medicaid and CHIP Child Enrollment
- Medic

## Clean State-Month Data

CMS provides both preliminary and updated/final records for many state-months. For the cleaned trend-analysis file, the workflow keeps one row per state-month and prefers final/updated records when both are available.

In [4]:
standardized = standardize_column_names(raw)
standardized["state_abbreviation"] = standardized["state_abbreviation"].astype(str).str.strip().str.upper()
standardized["reporting_month"] = standardized["reporting_period"].apply(parse_reporting_period)
standardized["reporting_year"] = standardized["reporting_month"].dt.year
filtered_raw = standardized[
    standardized["state_abbreviation"].isin(STATE_ABBREVIATIONS)
    & (standardized["reporting_year"] >= 2019)
]
source_duplicates = int(filtered_raw.duplicated(["state_abbreviation", "reporting_month"]).sum())

cleaned = clean_data(raw)
paths = save_outputs(cleaned, raw_row_count=len(raw), source_duplicate_count=source_duplicates)

print(f"Source duplicate state-month records before final-report selection: {source_duplicates:,}")
print(f"Cleaned rows: {len(cleaned):,}")
print(f"Cleaned columns: {len(cleaned.columns):,}")
print(f"Date range: {cleaned['reporting_month'].min().date()} to {cleaned['reporting_month'].max().date()}")
print(f"Reporting months: {cleaned['reporting_month'].nunique():,}")
print(f"States/DC included: {cleaned['state_abbreviation'].nunique():,}")
print(f"Cleaned duplicate state-month records: {cleaned.duplicated(['state_abbreviation', 'reporting_month']).sum():,}")
paths

Source duplicate state-month records before final-report selection: 4,335
Cleaned rows: 4,386
Cleaned columns: 27
Date range: 2019-01-01 to 2026-02-01
Reporting months: 86
States/DC included: 51
Cleaned duplicate state-month records: 0


{'cleaned': PosixPath('/Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard/data/processed/medicaid_enrollment_clean.csv'),
 'summary': PosixPath('/Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard/data/processed/medicaid_state_month_summary.csv'),
 'sample': PosixPath('/Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard/data/processed/medicaid_sample_for_dashboard.csv'),
 'quality': PosixPath('/Users/carolinehoward/Documents/Medicaid/medicaid-enrollment-policy-dashboard/data/processed/data_quality_summary.csv')}

## Field Availability

The selected source supports enrollment trend analysis and several eligibility operations measures. It does not provide individual-level details, claims, causal policy effects, or all operational concepts that might be useful, such as renewal/redetermination counts or pending applications in the preserved fields.

In [5]:
field_groups = {
    "state_fields": ["state_abbreviation", "state_name"],
    "date_fields": ["reporting_period", "reporting_month", "reporting_year"],
    "reporting_status_fields": ["preliminary_or_updated", "final_report"],
    "enrollment_fields": [
        "total_medicaid_and_chip_enrollment",
        "total_medicaid_enrollment",
        "total_chip_enrollment",
        "medicaid_and_chip_child_enrollment",
        "total_adult_medicaid_enrollment",
    ],
    "applications_determinations_fields": [
        "new_applications_submitted_to_medicaid_and_chip_agencies",
        "applications_for_financial_assistance_submitted_to_the_state_based_marketplace",
        "total_applications_for_financial_assistance_submitted_at_state_level",
        "individuals_determined_eligible_for_medicaid_at_application",
        "individuals_determined_eligible_for_chip_at_application",
        "total_medicaid_and_chip_determinations",
    ],
    "processing_timing_and_call_center_fields": [
        "total_medicaid_and_chip_determinations_processed_in_less_than_24_hours",
        "total_medicaid_and_chip_determinations_processed_between_24_hours_and_7_days",
        "total_medicaid_and_chip_determinations_processed_between_8_days_and_30_days",
        "total_medicaid_and_chip_determinations_processed_between_31_days_and_45_days",
        "total_medicaid_and_chip_determinations_processed_in_more_than_45_days",
        "total_call_center_volume_number_of_calls",
        "average_call_center_wait_time_minutes",
        "average_call_center_abandonment_rate",
    ],
}

for group_name, columns in field_groups.items():
    available = [column for column in columns if column in cleaned.columns]
    print(f"{group_name}: {len(available)} available")
    for column in available:
        print(f"  - {column}")

state_fields: 2 available
  - state_abbreviation
  - state_name
date_fields: 3 available
  - reporting_period
  - reporting_month
  - reporting_year
reporting_status_fields: 2 available
  - preliminary_or_updated
  - final_report
enrollment_fields: 5 available
  - total_medicaid_and_chip_enrollment
  - total_medicaid_enrollment
  - total_chip_enrollment
  - medicaid_and_chip_child_enrollment
  - total_adult_medicaid_enrollment
applications_determinations_fields: 6 available
  - new_applications_submitted_to_medicaid_and_chip_agencies
  - applications_for_financial_assistance_submitted_to_the_state_based_marketplace
  - total_applications_for_financial_assistance_submitted_at_state_level
  - individuals_determined_eligible_for_medicaid_at_application
  - individuals_determined_eligible_for_chip_at_application
  - total_medicaid_and_chip_determinations
processing_timing_and_call_center_fields: 8 available
  - total_medicaid_and_chip_determinations_processed_in_less_than_24_hours
  - tota

## Source Validation Checks

In [6]:
print("Reporting frequency: monthly state-level aggregate data")
print("Final report values:")
print(cleaned["final_report"].value_counts(dropna=False))
print("\nPreliminary or updated values:")
print(cleaned["preliminary_or_updated"].value_counts(dropna=False))

months_per_state = cleaned.groupby("state_abbreviation")["reporting_month"].nunique().describe()
print("\nMonths per state/DC:")
print(months_per_state)

print("\nTop missingness counts:")
print(cleaned.isna().sum().sort_values(ascending=False).head(15))

Reporting frequency: monthly state-level aggregate data
Final report values:
Y    4335
N      51
Name: final_report, dtype: int64

Preliminary or updated values:
U    4335
P      51
Name: preliminary_or_updated, dtype: int64

Months per state/DC:
count    51.0
mean     86.0
std       0.0
min      86.0
25%      86.0
50%      86.0
75%      86.0
max      86.0
Name: reporting_month, dtype: float64

Top missingness counts:
total_adult_medicaid_enrollment                                                 3366
average_call_center_abandonment_rate                                            2590
total_call_center_volume_number_of_calls                                        2590
average_call_center_wait_time_minutes                                           2589
total_medicaid_and_chip_determinations_processed_in_more_than_45_days           1377
total_medicaid_and_chip_determinations_processed_between_31_days_and_45_days    1377
total_medicaid_and_chip_determinations_processed_between_8_days_and_

In [7]:
quality = pd.read_csv(QUALITY_SUMMARY_PATH)
print("Quality summary rows by check type:")
print(quality["check_type"].value_counts())
quality.head(12)

Quality summary rows by check type:
state_month_missingness    4386
monthly_completeness         86
field_missingness            27
dataset_summary               2
duplicate_check               2
value_check                   2
Name: check_type, dtype: int64


,check_type,field_name,value,percent,notes
0,dataset_summary,raw_row_count,10710,NaN,Rows in the downloaded CMS source CSV before f...
1,dataset_summary,cleaned_row_count,4386,NaN,Rows after filtering to 2019-present and 50 st...
2,duplicate_check,source_state_abbreviation_reporting_month,4335,NaN,Duplicate state-month records in the source ex...
3,duplicate_check,cleaned_state_abbreviation_reporting_month,0,0.0,Duplicate count at the state-month grain after...
4,value_check,negative_numeric_values,0,NaN,Negative values across preserved numeric enrol...
5,value_check,zero_numeric_values,5244,NaN,Zero values across preserved numeric fields; z...
6,field_missingness,state_abbreviation,0,0.0,Missing values counted after filtering to 2019...
7,field_missingness,state_name,0,0.0,Missing values counted after filtering to 2019...
8,field_missingness,reporting_period,0,0.0,Missing values counted after filtering to 2019...
9,field_missingness,reporting_month,0,0.0,Missing values counted after filtering to 2019...


## Data Quality Concerns

- The source includes preliminary and updated/final records for many state-months. The cleaned file keeps one row per state-month and prefers final/updated records.
- The latest reporting month in this extract includes preliminary records for all states/DC, so interpretation should treat the latest month cautiously.
- Adult Medicaid enrollment, call center fields, and application-processing timing fields have substantial missingness and should be clearly labeled in any dashboard or policy brief.
- Zero values are present in numeric fields. Some zeros may be valid program/reporting values, while others may require state-specific source review.
- Public aggregate data supports descriptive monitoring, reporting, and operational analytics. It does not support beneficiary-level conclusions or causal claims without additional design and evidence.

## Processed Outputs Created

- `data/processed/medicaid_enrollment_clean.csv`: full cleaned 2019-present all-state/DC state-month dataset.
- `data/processed/medicaid_state_month_summary.csv`: compact state-month summary for analysis.
- `data/processed/medicaid_sample_for_dashboard.csv`: smaller recent-month sample for quick testing and dashboard development.
- `data/processed/data_quality_summary.csv`: field, duplicate, monthly completeness, and state-month missingness checks.

Next project step: exploratory analysis and dashboard-ready summary tables. Do not build the dashboard until the EDA and summary table step is complete.